
## **1. Imports**


In [ ]:
!pip install pandas requests tqdm

In [ ]:
import os
from pathlib import Path

def _load_key_from_env_file(env_path: Path, key_name: str = "OPENROUTER_API_KEY") -> str:
    if not env_path.exists():
        return ""
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if k.strip() == key_name:
            return v.strip().strip('"').strip("'")
    return ""

ENV_PATH = Path("eval/.env")
if not ENV_PATH.exists():
    ENV_PATH = Path("eval.env")
if not ENV_PATH.exists():
    ENV_PATH = Path("../../.env")

OPENROUTER_API_KEY = _load_key_from_env_file(ENV_PATH) or os.environ.get("OPENROUTER_API_KEY", "")
assert OPENROUTER_API_KEY, f"OPENROUTER_API_KEY was not found in {ENV_PATH} or environment variables"
print(f"OPENROUTER_API_KEY loaded ({'env file' if _load_key_from_env_file(ENV_PATH) else 'environment'})")

In [ ]:
import json

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

MODEL = "anthropic/claude-opus-4.6"   # or "anthropic/claude-opus-4.6"

# I/O
INPUT_CSV = "dataset/2_rules_understanding.csv"
OUTPUT_CSV = "2_1-2_with_answers_claude-opus_4_6.csv"
OUTPUT_JSONL = "2_1-2_with_answers_claude-opus_4_6_raw.jsonl"

# Column in CSV that contains the prompt/question
QUESTION_COL = "question_text"
VIEWPOINT_SCENES_COL = "figure_path"  # optional image column for this dataset


# Generation params
MAX_TOKENS = 500
RETRY_MAX_TOKENS = 1000
TEMPERATURE = 0.0

## 3. OpenRouter call utilities

In [ ]:
import base64
import mimetypes
import time
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import requests


def load_input_dataframe(input_path: str) -> pd.DataFrame:
    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f"Input dataset not found: {p}")

    suffix = p.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(p)
    if suffix == ".jsonl":
        return pd.read_json(p, lines=True)
    if suffix == ".json":
        try:
            return pd.read_json(p)
        except ValueError:
            return pd.read_json(p, lines=True)

    raise ValueError(f"Unsupported input format: {suffix}. Use .csv, .json or .jsonl")


def build_headers(api_key: str) -> Dict[str, str]:
    return {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }


def check_key_info(api_key: str) -> Dict[str, Any]:
    """GET /key - check usage/credits for the key."""
    r = requests.get(f"{OPENROUTER_BASE_URL}/key", headers=build_headers(api_key), timeout=30)
    r.raise_for_status()
    return r.json()


def build_prompt(question_text: str) -> str:
    q = "" if question_text is None else str(question_text).strip()
    return q


def normalize_image_value(v: Any) -> Optional[str]:
    if v is None:
        return None
    try:
        import pandas as _pd
        if _pd.isna(v):
            return None
    except Exception:
        pass
    s = str(v).strip().strip('"').strip("'")
    return s if s else None


def collect_image_values(*values: Any) -> list[str]:
    out: list[str] = []
    seen = set()
    for v in values:
        s = normalize_image_value(v)
        if not s:
            continue
        if s in seen:
            continue
        out.append(s)
        seen.add(s)
    return out


def is_image_ref(s: str) -> bool:
    if not s:
        return False
    return Path(s).exists()


def to_image_url(image_value: str) -> str:
    s = "" if image_value is None else str(image_value).strip().strip('"').strip("'")
    if not s:
        raise ValueError("Empty image value")

    p = Path(s)
    if not p.exists():
        raise FileNotFoundError(s)

    mime, _ = mimetypes.guess_type(p.name)
    if not mime:
        mime = "image/jpeg"

    data = p.read_bytes()
    b64 = base64.b64encode(data).decode("ascii")
    return f"data:{mime};base64,{b64}"


def call_openrouter_chat(
    api_key: str,
    prompt: str,
    image_values: Optional[list[str]],
    request_id: str | None = None,
    max_tokens: int | None = None,
) -> dict:
    url = f"{OPENROUTER_BASE_URL}/chat/completions"

    prompt_final = build_prompt(prompt)
    content = [{"type": "text", "text": prompt_final}]

    for image_value in (image_values or []):
        if not image_value:
            continue
        if not is_image_ref(image_value):
            continue
        image_url = to_image_url(image_value)
        content.append({"type": "image_url", "image_url": {"url": image_url}})


    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": content,
            }
        ],
        "max_tokens": MAX_TOKENS if max_tokens is None else int(max_tokens),
        "temperature": TEMPERATURE,
    }

    if request_id is not None:
        payload["metadata"] = {"request_id": request_id}

    r = requests.post(url, headers=build_headers(api_key), json=payload, timeout=120)

    if r.status_code >= 400:
        try:
            err = r.json()
        except Exception:
            err = {"error": {"message": r.text}}
        raise RuntimeError(f"OpenRouter error {r.status_code}: {err}")

    return r.json()


def extract_finish_reason(resp: Dict[str, Any]) -> str:
    try:
        return str(resp["choices"][0].get("finish_reason", "")).strip().lower()
    except Exception:
        return ""


def extract_content(resp: Dict[str, Any]) -> Any:
    try:
        return resp["choices"][0]["message"].get("content")
    except Exception:
        return None


def response_needs_retry(resp: Dict[str, Any]) -> bool:
    finish_reason = extract_finish_reason(resp)
    content = extract_content(resp)

    if finish_reason == "length":
        return True
    if content is None:
        return True
    if isinstance(content, str) and not content.strip():
        return True
    if isinstance(content, list) and len(content) == 0:
        return True

    return False


def call_with_retries(
    api_key: str,
    prompt: str,
    image_values: Optional[list[str]],
    request_id: str,
    max_attempts: int = 3,
    retry_max_tokens: int = RETRY_MAX_TOKENS,
) -> Dict[str, Any]:
    """Retry with larger token limit from the 2nd attempt onward."""
    if max_attempts < 1:
        raise ValueError("max_attempts must be >= 1")

    backoff = 2.0

    for attempt in range(1, max_attempts + 1):
        attempt_max_tokens = MAX_TOKENS if attempt == 1 else retry_max_tokens
        try:
            resp = call_openrouter_chat(
                api_key=api_key,
                prompt=prompt,
                image_values=image_values,
                request_id=request_id,
                max_tokens=attempt_max_tokens,
            )
        except RuntimeError as e:
            msg = str(e)
            retriable = any(code in msg for code in [" 429:", " 502:", " 503:", " 408:"])
            if (attempt == max_attempts) or (not retriable):
                raise
            time.sleep(backoff)
            backoff *= 1.7
            continue

        if response_needs_retry(resp):
            if attempt == max_attempts:
                raise RuntimeError(f"Text response incomplete after {max_attempts} attempts")
            time.sleep(backoff)
            backoff *= 1.7
            continue

        return resp

    raise RuntimeError("Retry loop exhausted unexpectedly")


def extract_text(resp: Dict[str, Any]) -> str:
    """choices[0].message.content"""
    c = extract_content(resp)
    return c if isinstance(c, str) else ""




In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv("dataset/1_1_full_dataset(1).csv")

def clean_path(s):
    s = str(s).strip().strip('"').strip("'")
    return s

missing = []
missing_viewpoint = []
for i, row in df.head(50).iterrows():
    image_values = collect_image_values(row.get(VIEWPOINT_SCENES_COL))
    viewpoint_value = normalize_image_value(row.get(VIEWPOINT_SCENES_COL))
    if (not viewpoint_value) or (not is_image_ref(viewpoint_value)):
        missing_viewpoint.append(i)
    for s in image_values:
        p = Path(clean_path(s))
        if not p.exists():
            missing.append((i, s))

missing[:5], len(missing), missing_viewpoint[:5], len(missing_viewpoint)


In [ ]:
info = check_key_info(OPENROUTER_API_KEY)
info

## 4. Run CSV -> model answers -> save
We save:
- `OUTPUT_CSV` ? input columns + `model`, `openrouter_request_id`, `model_answer`
- `OUTPUT_JSONL` ? raw responses (useful for debugging/auditing)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, Image

INPUT_CSV="dataset/2_rules_understanding.csv"
df = pd.read_csv(INPUT_CSV)

PREVIEW_ROW = 10

row = df.iloc[PREVIEW_ROW]

question = str(row[QUESTION_COL])
image_values = collect_image_values(row.get(VIEWPOINT_SCENES_COL))

prompt_final = build_prompt(question)

print("=== PREVIEW: prompt_final ===")
print(prompt_final)
print("\n=== PREVIEW: image_values ===")
print(image_values)

for idx, image_value in enumerate(image_values):
    p = Path(image_value.strip().strip('"').strip("'"))
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f"Image {idx} file not found locally.")


In [ ]:
COL_TEMPLATE = "template_id"

In [ ]:
{VIEWPOINT_SCENES_COL: int(df[VIEWPOINT_SCENES_COL].isna().sum())}, len(df)
missing_viewpoint = df[df[VIEWPOINT_SCENES_COL].isna()]
missing_viewpoint[[COL_TEMPLATE, QUESTION_COL, VIEWPOINT_SCENES_COL]].head(10)


In [ ]:
missing_viewpoint = df[df[VIEWPOINT_SCENES_COL].isna()]
print("Rows with NaN in viewpoint scenes path:", missing_viewpoint.index.tolist()[:50], "…")
print("Count:", len(missing_viewpoint))

missing_viewpoint[[QUESTION_COL, VIEWPOINT_SCENES_COL]].head(10)


### Preview: 6 random questions (3 per layer_id)

In [ ]:
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import json

PREVIEW_PER_LAYER = 2
PREVIEW_LAYER_IDS = ["2_1", "2_2"]
PREVIEW_SEED = 60

PREVIEW_CSV = "2_1-2_preview_claude-opus_4_6.csv"
PREVIEW_JSONL = "2_1-2_preview_claude-opus_4_6_raw.jsonl"

df_preview = load_input_dataframe(INPUT_CSV)

assert QUESTION_COL in df_preview.columns, (
    f"Input must contain column '{QUESTION_COL}'. Found: {list(df_preview.columns)}"
)
if VIEWPOINT_SCENES_COL not in df_preview.columns:
    print(
        f"Column '{VIEWPOINT_SCENES_COL}' is absent in input; preview will run in text-only mode."
    )
    df_preview[VIEWPOINT_SCENES_COL] = ""
assert "layer_id" in df_preview.columns, "Input must contain column 'layer_id'"

df_preview = df_preview.copy()
df_preview["_layer_id_str"] = df_preview["layer_id"].astype(str).str.strip()


# If a preview already exists, do not overwrite it.
# At the same time, exclude questions that have already been viewed from the new selection,
# so as not to duplicate them again.
existing_preview_df = pd.DataFrame()

if Path(PREVIEW_CSV).exists():
    existing_preview_df = pd.read_csv(PREVIEW_CSV)

    if "generated_question_id" in df_preview.columns and "generated_question_id" in existing_preview_df.columns:
        existing_ids = set(existing_preview_df["generated_question_id"].dropna().astype(str))
        before = len(df_preview)
        df_preview = df_preview[
            ~df_preview["generated_question_id"].astype(str).isin(existing_ids)
        ].copy()
        print(f"Skipping {before - len(df_preview)} rows already present in {PREVIEW_CSV}.")
    elif "row_index" in existing_preview_df.columns:
        existing_row_indices = set(
            pd.to_numeric(existing_preview_df["row_index"], errors="coerce")
            .dropna()
            .astype(int)
        )
        before = len(df_preview)
        df_preview = df_preview[~df_preview.index.isin(existing_row_indices)].copy()
        print(f"Skipping {before - len(df_preview)} rows already present in {PREVIEW_CSV} (by row_index).")

sample_parts = []

for offset, layer_id in enumerate(PREVIEW_LAYER_IDS):
    layer_df = df_preview[df_preview["_layer_id_str"] == str(layer_id)].copy()
    n_take = min(PREVIEW_PER_LAYER, len(layer_df))

    if n_take == 0:
        print(f"No available rows for layer_id={layer_id}")
        continue

    layer_sample = layer_df.sample(
        n=n_take,
        random_state=PREVIEW_SEED + offset
    )
    sample_parts.append(layer_sample)

if not sample_parts:
    raise ValueError("No rows available for preview after filtering existing preview rows.")

sample_df = pd.concat(sample_parts, axis=0).sort_index()

print(
    f"Preview sample size: {len(sample_df)} | "
    + " | ".join(
        f"layer {layer_id}: {sum(sample_df['_layer_id_str'] == str(layer_id))}"
        for layer_id in PREVIEW_LAYER_IDS
    )
)

preview_rows = []
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

for i, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    prompt = str(row[QUESTION_COL])
    image_value = normalize_image_value(row.get(VIEWPOINT_SCENES_COL))
    image_exists = bool(image_value) and is_image_ref(image_value)
    image_values = collect_image_values(image_value) if image_exists else []
    request_id = f"preview_{run_id}_row_{i}"
    generated_question_id = row.get("generated_question_id")

    warning = ""
    if not image_value:
        warning = f"Image path is missing in column '{VIEWPOINT_SCENES_COL}'. Sent text-only prompt."
    elif not image_exists:
        warning = f"Image path not found: {image_value}. Sent text-only prompt."

    error_reason = ""
    text = ""
    try:
        resp = call_with_retries(
            OPENROUTER_API_KEY,
            prompt,
            image_values,
            request_id,
            max_attempts=3,
        )
        text = extract_text(resp)
        error_reason = warning
    except RuntimeError as e:
        error_reason = f"{warning} | {e}" if warning else str(e)

    preview_rows.append({
        "preview_run_id": run_id,
        "row_index": i,
        "generated_question_id": generated_question_id,
        "layer_id": row.get("layer_id"),
        "scene_id": row.get("scene_id"),
        "rule_figure_id": row.get("rule_figure_id"),
        "question_text": prompt,
        "correct_answer": row.get("ground_truth_answer"),
        "model_answer": text,
        "openrouter_request_id": request_id,
        "error_reason": error_reason,
    })

new_preview_df = pd.DataFrame(preview_rows).sort_values("row_index")

if Path(PREVIEW_CSV).exists():
    old_preview_df = pd.read_csv(PREVIEW_CSV)
    preview_df = pd.concat([old_preview_df, new_preview_df], ignore_index=True)
else:
    preview_df = new_preview_df.copy()

preview_df.to_csv(PREVIEW_CSV, index=False)

jsonl_rows_df = new_preview_df.where(pd.notna(new_preview_df), None)
with open(PREVIEW_JSONL, "a", encoding="utf-8") as f:
    for record in jsonl_rows_df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\\n")

with open(PREVIEW_JSONL, "r", encoding="utf-8") as f:
    total_jsonl_rows = sum(1 for _ in f)

print(f"Added {len(new_preview_df)} new rows to: {PREVIEW_CSV}")
print(f"Total rows in CSV now: {len(preview_df)}")
print(f"Appended {len(new_preview_df)} rows to: {PREVIEW_JSONL}")
print(f"Total rows in JSONL now: {total_jsonl_rows}")

new_preview_df

#### table fin

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent.parent.parent

MAIN_DATASET_PATH = ROOT / "dataset" / "2_rules_understanding.csv"
PREVIEW_PATH = ROOT / "2_1-2_preview_claude-opus_4_6.csv"
OUTPUT_PATH = Path(OUTPUT_CSV)

main_df = pd.read_csv(MAIN_DATASET_PATH)
preview_df = pd.read_csv(PREVIEW_PATH)

assert "generated_question_id" in main_df.columns, "Main dataset must contain 'generated_question_id'"
assert "generated_question_id" in preview_df.columns, "Preview file must contain 'generated_question_id'"
assert "model_answer" in preview_df.columns, "Preview file must contain 'model_answer'"


preview_answers = preview_df[["generated_question_id", "model_answer"]].copy()

preview_answers = preview_answers.dropna(subset=["generated_question_id"])
preview_answers = preview_answers.drop_duplicates(subset=["generated_question_id"], keep="last")

main_df["generated_question_id"] = main_df["generated_question_id"].astype(str)
preview_answers["generated_question_id"] = preview_answers["generated_question_id"].astype(str)

answer_map = dict(zip(preview_answers["generated_question_id"], preview_answers["model_answer"]))

if "model_answer" not in main_df.columns:
    main_df["model_answer"] = pd.NA

matched_mask = main_df["generated_question_id"].isin(answer_map.keys())
main_df.loc[matched_mask, "model_answer"] = main_df.loc[matched_mask, "generated_question_id"].map(answer_map)

main_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_PATH}")
print(f"Matched rows updated: {matched_mask.sum()}")

### Run the dataset on the model:

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json

FULL_INPUT = Path(INPUT_CSV)
OUTPUT_PATH = Path(OUTPUT_CSV)
OUTPUT_JSONL = "2_1-2_with_answers_claude-opus_4_6_raw.jsonl"


def _normalize_gid_series(s: pd.Series) -> pd.Series:
    x = s.astype(str).str.strip()
    x = x.str.replace(r"\.0$", "", regex=True)  # 3.0 -> 3
    return x


def _is_empty_answer(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip().str.lower()
    return s.isin(["", "0", "nan", "none", "null"])


assert FULL_INPUT.exists(), f"Input dataset not found: {FULL_INPUT}"

df_full = load_input_dataframe(str(FULL_INPUT))
assert QUESTION_COL in df_full.columns, f"Input must contain column '{QUESTION_COL}'. Found: {list(df_full.columns)}"
if VIEWPOINT_SCENES_COL not in df_full.columns:
    print(
        f"Column '{VIEWPOINT_SCENES_COL}' is absent in input; dataset run will proceed in text-only mode."
    )
    df_full[VIEWPOINT_SCENES_COL] = ""
assert "generated_question_id" in df_full.columns, "Input must contain generated_question_id"
assert "layer_id" in df_full.columns, "Input must contain layer_id"
df_full["_layer_id_str"] = df_full["layer_id"].astype(str).str.strip()
df_full = df_full[df_full["_layer_id_str"].isin(["2_1", "2_2"])].copy()
df_full = df_full.drop(columns=["_layer_id_str"], errors="ignore")

df_full["_gid_norm"] = _normalize_gid_series(df_full["generated_question_id"])

payload_cols = [
    "model_answer",
    "openrouter_request_id",
    "error_reason",
    "row_index",
    "bool_model_answer",
    "model_bool_answer",
    "parsed_json",
    "json_valid",
    "model_JSON_answer",
    "model",
]

if OUTPUT_PATH.exists():
    existing_raw = pd.read_csv(OUTPUT_PATH)
else:
    existing_raw = pd.DataFrame(columns=["generated_question_id"] + payload_cols)

for c in ["generated_question_id"] + payload_cols:
    if c not in existing_raw.columns:
        existing_raw[c] = ""

existing_raw["_gid_norm"] = _normalize_gid_series(existing_raw["generated_question_id"])
existing_raw["_has_answer"] = (~_is_empty_answer(existing_raw["model_answer"])).astype(int)

existing_best = (
    existing_raw.sort_values(["_has_answer"], ascending=False)
    .drop_duplicates(subset=["_gid_norm"], keep="first")
)

# Always rebuild OUTPUT_CSV on top of full input while preserving existing non-empty answers.
df = df_full.merge(
    existing_best[["_gid_norm"] + payload_cols],
    on="_gid_norm",
    how="left",
    suffixes=("", "_old"),
)

for c in payload_cols:
    if c not in df.columns:
        df[c] = ""

def _json_default(o):
    if hasattr(o, "item"):
        try:
            return o.item()
        except Exception:
            pass
    try:
        if pd.isna(o):
            return None
    except Exception:
        pass
    return str(o)

SAVE_EVERY = 10
pending_idx = df.index[_is_empty_answer(df["model_answer"])].tolist()

print(f"Total rows in input: {len(df)}")
print(f"Rows already answered: {len(df) - len(pending_idx)}")
print(f"Rows pending model call: {len(pending_idx)}")

appended_jsonl_rows = 0

with open(OUTPUT_JSONL, "a", encoding="utf-8") as fjsonl:
    for step, idx in enumerate(tqdm(pending_idx), start=1):
        row = df.loc[idx]
        prompt = str(row[QUESTION_COL])
        image_value = normalize_image_value(row.get(VIEWPOINT_SCENES_COL))
        image_exists = bool(image_value) and is_image_ref(image_value)
        image_values = collect_image_values(image_value) if image_exists else []
        request_id = f"row_{idx}"
        gid = row.get("generated_question_id")

        warning = ""
        if not image_value:
            warning = f"Image path is missing in column '{VIEWPOINT_SCENES_COL}'. Sent text-only prompt."
        elif not image_exists:
            warning = f"Image path not found: {image_value}. Sent text-only prompt."

        try:
            resp = call_with_retries(
                OPENROUTER_API_KEY,
                prompt,
                image_values,
                request_id,
                max_attempts=3,
            )
            text = extract_text(resp)
            df.at[idx, "model_answer"] = text
            df.at[idx, "openrouter_request_id"] = request_id
            df.at[idx, "error_reason"] = warning
            df.at[idx, "row_index"] = idx
            df.at[idx, "model"] = MODEL
            fjsonl.write(json.dumps(
                {"row_index": idx, "generated_question_id": gid, "request_id": request_id, "response": resp},
                ensure_ascii=False,
                default=_json_default,
            ) + "\n")
            appended_jsonl_rows += 1
        except RuntimeError as e:
            reason = f"{warning} | {e}" if warning else str(e)
            df.at[idx, "model_answer"] = ""
            df.at[idx, "openrouter_request_id"] = request_id
            df.at[idx, "error_reason"] = reason
            df.at[idx, "row_index"] = idx
            df.at[idx, "model"] = MODEL
            fjsonl.write(json.dumps(
                {"row_index": idx, "generated_question_id": gid, "request_id": request_id, "error": reason},
                ensure_ascii=False,
                default=_json_default,
            ) + "\n")
            appended_jsonl_rows += 1

        if step % SAVE_EVERY == 0:
            df.drop(columns=["_gid_norm"], errors="ignore").to_csv(OUTPUT_PATH, index=False)

df_out = df.drop(columns=["_gid_norm"], errors="ignore").copy()
df_out.to_csv(OUTPUT_PATH, index=False)

remaining_empty = int(_is_empty_answer(df_out["model_answer"]).sum())
with open(OUTPUT_JSONL, "r", encoding="utf-8") as fjsonl:
    total_jsonl_rows = sum(1 for _ in fjsonl)
print("Saved OUTPUT_CSV:", OUTPUT_PATH)
print("Saved raw JSONL:", OUTPUT_JSONL)
print("Appended rows to raw JSONL:", appended_jsonl_rows)
print("Total rows in raw JSONL:", total_jsonl_rows)
print("Rows in output:", len(df_out))
print("Remaining empty model_answer:", remaining_empty)

df = df_out.copy()
df_out.head(3)


### Sanity check:

In [ ]:
import numpy as np

lens = df["model_answer"].fillna("").astype(str).map(len)
print("Rows:", len(df))
print("Empty answers:", int((lens == 0).sum()))
print("Median length:", int(np.median(lens)))
print("P90 length:", int(np.quantile(lens, 0.9)))

# 5. Metrics

In [ ]:
import pandas as pd

file = "2_1-2_with_answers_claude-opus_4_6.csv"
cols_to_drop = ["model_bool_answer", "parsed_json", "json_valid", "model_JSON_answer"]

df = pd.read_csv(file)
df.drop(columns=cols_to_drop, errors="ignore").to_csv(file, index=False)

### Extract Yes/No answers:

In [ ]:
import pandas as pd
import re

INPUT = "1_1_with_answers_claude-opus_4_6.csv"

df = pd.read_csv(INPUT)

def extract_yes_no(text: str) -> str:
    s = "" if pd.isna(text) else str(text)
    # Find explicit @YES/@NO marker, prefer the last one if multiple
    matches = re.findall(r"@\s*(YES|NO)\b", s, flags=re.IGNORECASE)
    if not matches:
        return ""
    return matches[-1].lower()

df["bool_model_answer"] = df["model_answer"].apply(extract_yes_no)

df.to_csv(INPUT, index=False)
print("Updated:", INPUT)
df[["model_answer", "bool_model_answer"]].head()


### F1&Accuracy

In [ ]:
import pandas as pd
import re

#INPUT = "output_with_answers_recovered.csv"
INPUT = "1_1_with_answers_claude-opus_4_6.csv"
#INPUT = "1_1_1_Eval_Claude_Opus_1.csv"

COL_TEMPLATE = "template_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

df = pd.read_csv(INPUT)

def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t

def contains_whole_word(text: str, word: str) -> bool:
    if not word:
        return False
    pattern = r"(?:^|\s)" + re.escape(word) + r"(?:$|\s)"
    return re.search(pattern, text) is not None

df["gt_norm"] = df[COL_GT].apply(normalize_label)
df["pred_norm_text"] = df[COL_PRED].apply(normalize_text)

def predicted_label_from_text(pred_text: str, gt_label: str) -> str:
    return gt_label if contains_whole_word(pred_text, gt_label) else ""

df["pred_label"] = [
    predicted_label_from_text(p, g)
    for p, g in zip(df["pred_norm_text"], df["gt_norm"])
]

# correctness
df["is_correct"] = df["pred_label"] == df["gt_norm"]

# ---- metrics per each template_id ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    labels = sorted(set(g["gt_norm"].unique()) | set(g["pred_label"].unique()))
    labels = [lab for lab in labels if lab != ""]  # уберём пустой

    # F1:
    # - If only yes/no in GT -> binary F1 (positive=yes)
    # - Otherwise, macro F1 by GT classes (ignoring empty class)
    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = [("yes" if x == "yes" else "no") if x in {"yes","no"} else "no" for x in g["pred_label"].tolist()]
        # F1 positive=yes
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        # macro F1 per GT classes
        y_true = g["gt_norm"].tolist()
        y_pred = g["pred_label"].tolist()

        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_type": f1_type,
    })

rows = []
for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
    m = group_metrics(g)          # Series: n, accuracy, f1, f1_type
    m[COL_TEMPLATE] = template_id
    rows.append(m)

report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)
report
